# Lab 2: Heterogeneous graphs (Amazon product co-purchasing network)

**Task 1: Product classification with heterogeneous graph neural networks**

The goal is to predict the **group** of each product (`Book`, `DVD`, `Music` CD or `Video`) with a heterogeneous GNN built in PyTorch Geometric.

The heterogeneous graph has:
* **Node types:** `product` and `customer`
* **Edge types:**
  * `product --similar--> product`: a product connects to the products listed as *similar* to it
  * `customer --rates--> product`: a customer connects to every product they rated. The **rating (1-5) is stored as an edge attribute** (`edge_attr`)
  * the reverse relations (`product --rev_rates--> customer`), so messages also flow from products to customers

We split the product nodes into a training set and a test set, and evaluate with `accuracy_score`, `precision_score`, `recall_score` and `f1_score`. Finally we try several hyperparameter configurations (neighbor sampling and training).

In [1]:
import torch
print(torch.__version__)

2.12.1+cu130


In [2]:
!pip install pyg_lib torch_cluster torch_sparse -f https://data.pyg.org/whl/torch-2.11.0+cu130.html
!pip install torch_geometric

Defaulting to user installation because normal site-packages is not writeable
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu130.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 693.2 kB/s  0:00:07m0:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 3.2 MB/s  0:00:01 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 4.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torch_cluster]0m [torch_sparse]

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import time
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.optim import Adam
from torch.nn import CrossEntropyLoss
from torch.nn.functional import dropout

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, GATConv, Linear, to_hetero
from torch_geometric.loader import NeighborLoader
from torch_geometric.utils import to_undirected

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

/home/kuzemano/.local/lib/python3.13/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /home/kuzemano/.local/lib/python3.13/site-packages/pyg_lib/libpyg.so
  import torch_geometric.typing
/home/kuzemano/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

## 1. Loading the dataset

`amazon-meta.txt` (from [SNAP](https://snap.stanford.edu/data/amazon-meta.html)) is a text file with one block per product:

```
Id:   1
ASIN: 0827229534
  title: Patterns of Preaching: A Sermon Sampler
  group: Book
  salesrank: 396585
  similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X
  categories: 2
   |Books[283155]|Subjects[1000]|...
  reviews: total: 2  downloaded: 2  avg rating: 5
    2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9
```

We parse the file line by line and keep only products from the four target groups. The few other groups (Toy, CE, Software, ...) and discontinued products are dropped. From each product we keep the ASIN, the group (the label), the sales rank, the list of similar products, the number of categories, the review summary and every individual review as a `(customer, rating)` pair.

In [5]:
DATA_PATH = 'amazon-meta.txt'
GROUPS = ['Book', 'DVD', 'Music', 'Video']


def parse_amazon(path, groups=GROUPS):
    products = []
    cur = None

    def flush(p):
        if p is not None and p.get('group') in groups:
            products.append(p)

    with open(path, encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line.startswith('Id:'):
                flush(cur)
                cur = {'similar': [], 'reviews': [], 'salesrank': -1,
                       'n_categories': 0, 'total_reviews': 0, 'avg_rating': 0.0}
            elif cur is None:
                continue
            elif line.startswith('ASIN:'):
                cur['asin'] = line[5:].strip()
            elif line.startswith('group:'):
                cur['group'] = line[6:].strip()
            elif line.startswith('salesrank:'):
                cur['salesrank'] = int(line[10:])
            elif line.startswith('similar:'):
                cur['similar'] = line.split()[2:]
            elif line.startswith('categories:'):
                cur['n_categories'] = int(line.split()[1])
            elif line.startswith('reviews:'):
                parts = line.split()  # reviews: total: 2 downloaded: 2 avg rating: 5
                cur['total_reviews'] = int(parts[2])
                cur['avg_rating'] = float(parts[-1])
            elif 'cutomer:' in line:
                parts = line.split()  # date cutomer: ID rating: R votes: V helpful: H
                cur['reviews'].append((parts[2], int(parts[4])))
    flush(cur)
    return products


start = time.time()
products = parse_amazon(DATA_PATH)
print(f'Parsed {len(products)} products in {time.time() - start:.1f}s')

Parsed 476137 products in 16.6s


In [6]:
group_counts = Counter(p['group'] for p in products)
print(group_counts)
print('Total reviews:', sum(len(p['reviews']) for p in products))

Counter({'Book': 349264, 'Music': 90895, 'Video': 22934, 'DVD': 13044})
Total reviews: 6329737


## 2. Building the heterogeneous graph

**Product nodes.** The label `y` is the group index. The node features are numeric metadata that do not reveal the group directly:
`log(1 + salesrank)`, a flag for a missing sales rank, the number of similar products, the number of categories, `log(1 + total reviews)`, `log(1 + downloaded reviews)` and the average rating. All features are standardized. The category strings (e.g. `|Books[283155]|...`) are **not** used, because their root category reveals the label.

**Customer nodes.** The dataset has no customer attributes, so we use structural features (feature augmentation): `log(1 + number of ratings)` and the customer's mean rating.

**Edges.**
* `('product', 'similar', 'product')`: only between products that are in the graph. The relation is made undirected.
* `('customer', 'rates', 'product')` with `edge_attr` = rating, plus the reverse `('product', 'rev_rates', 'customer')` with the same attribute.

In [7]:
asin_to_idx = {p['asin']: i for i, p in enumerate(products)}
group_to_idx = {g: i for i, g in enumerate(GROUPS)}
num_products = len(products)

# --- product -> product (similar) ---
sim_src, sim_dst = [], []
for i, p in enumerate(products):
    for s in p['similar']:
        j = asin_to_idx.get(s)
        if j is not None and j != i:
            sim_src.append(i)
            sim_dst.append(j)

# --- customer -> product (rates) ---
customer_to_idx = {}
rev_cust, rev_prod, rev_rating = [], [], []
for i, p in enumerate(products):
    for cust, rating in p['reviews']:
        c = customer_to_idx.setdefault(cust, len(customer_to_idx))
        rev_cust.append(c)
        rev_prod.append(i)
        rev_rating.append(rating)
num_customers = len(customer_to_idx)

print('products:', num_products, 'customers:', num_customers)
print('similar edges:', len(sim_src), 'rating edges:', len(rev_cust))

products: 476137 customers: 1434903
similar edges: 922128 rating edges: 6329737


In [8]:
def standardize(x):
    return (x - x.mean(0, keepdim=True)) / (x.std(0, keepdim=True) + 1e-8)


# --- product features and labels ---
salesrank = np.array([p['salesrank'] for p in products], dtype=np.float32)
product_x = np.stack([
    np.log1p(np.clip(salesrank, 0, None)),
    (salesrank <= 0).astype(np.float32),
    np.array([len(p['similar']) for p in products], dtype=np.float32),
    np.array([p['n_categories'] for p in products], dtype=np.float32),
    np.log1p(np.array([p['total_reviews'] for p in products], dtype=np.float32)),
    np.log1p(np.array([len(p['reviews']) for p in products], dtype=np.float32)),
    np.array([p['avg_rating'] for p in products], dtype=np.float32),
], axis=1)
product_x = standardize(torch.tensor(product_x))
product_y = torch.tensor([group_to_idx[p['group']] for p in products], dtype=torch.long)

# --- customer features (structural) ---
rev_cust_t = torch.tensor(rev_cust, dtype=torch.long)
rev_prod_t = torch.tensor(rev_prod, dtype=torch.long)
rev_rating_t = torch.tensor(rev_rating, dtype=torch.float)

cust_deg = torch.bincount(rev_cust_t, minlength=num_customers).float()
cust_rating_sum = torch.zeros(num_customers).scatter_add_(0, rev_cust_t, rev_rating_t)
customer_x = standardize(torch.stack([torch.log1p(cust_deg), cust_rating_sum / cust_deg.clamp(min=1)], dim=1))

product_x.shape, customer_x.shape

(torch.Size([476137, 7]), torch.Size([1434903, 2]))

In [9]:
data = HeteroData()

data['product'].x = product_x
data['product'].y = product_y
data['customer'].x = customer_x

# similar products (undirected); constant edge attribute so every relation has an edge_attr
sim_edge_index = to_undirected(torch.tensor([sim_src, sim_dst], dtype=torch.long), num_nodes=num_products)
data['product', 'similar', 'product'].edge_index = sim_edge_index
data['product', 'similar', 'product'].edge_attr = torch.ones(sim_edge_index.size(1), 1)

# customer rates product, with the rating as an edge attribute
rates_edge_index = torch.stack([rev_cust_t, rev_prod_t], dim=0)
rating_attr = rev_rating_t.view(-1, 1)
data['customer', 'rates', 'product'].edge_index = rates_edge_index
data['customer', 'rates', 'product'].edge_attr = rating_attr

# reverse relation, so information also flows product -> customer -> product
data['product', 'rev_rates', 'customer'].edge_index = rates_edge_index.flip(0)
data['product', 'rev_rates', 'customer'].edge_attr = rating_attr.clone()

del products, sim_src, sim_dst, rev_cust, rev_prod, rev_rating  # free memory
data

HeteroData(
  product={
    x=[476137, 7],
    y=[476137],
  },
  customer={ x=[1434903, 2] },
  (product, similar, product)={
    edge_index=[2, 1476386],
    edge_attr=[1476386, 1],
  },
  (customer, rates, product)={
    edge_index=[2, 6329737],
    edge_attr=[6329737, 1],
  },
  (product, rev_rates, customer)={
    edge_index=[2, 6329737],
    edge_attr=[6329737, 1],
  }
)

In [10]:
data.metadata()

(['product', 'customer'],
 [('product', 'similar', 'product'),
  ('customer', 'rates', 'product'),
  ('product', 'rev_rates', 'customer')])

In [11]:
data.validate()

True

## 3. Train / test split

We split the product nodes into **80% training** and **20% test**, stratified by group, because the classes are heavily imbalanced (most products are books). Customer nodes are not labeled, so they only pass messages.

In [12]:
all_idx = np.arange(num_products)
train_idx, test_idx = train_test_split(all_idx, test_size=0.2, stratify=product_y.numpy(), random_state=42)

data['product'].train_mask = torch.zeros(num_products, dtype=torch.bool)
data['product'].test_mask = torch.zeros(num_products, dtype=torch.bool)
data['product'].train_mask[train_idx] = True
data['product'].test_mask[test_idx] = True

print('train:', data['product'].train_mask.sum().item(), 'test:', data['product'].test_mask.sum().item())
print('train class distribution:', torch.bincount(data['product'].y[data['product'].train_mask]).tolist())

train: 380909 test: 95228
train class distribution: [279411, 10435, 72716, 18347]


## 4. Models

We follow the pattern from the exercises: write a homogeneous GNN and convert it to a heterogeneous one with `to_hetero`. This creates separate weights for each relation (`similar`, `rates`, `rev_rates`) and aggregates the messages per node type.

* **GraphSAGE** (`SAGEConv((-1, -1), ...)`) as in the exercises. It ignores edge attributes.
* **GAT** (`GATConv((-1, -1), ..., edge_dim=1)`), which **uses the rating as an edge attribute** when it computes the attention coefficients.

In [13]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, num_classes, hidden=64, p=0.3):
        super().__init__()
        self.p = p

        self.conv1 = SAGEConv((-1, -1), hidden)
        self.conv2 = SAGEConv((-1, -1), hidden * 2)

        self.linear1 = Linear(hidden * 2, hidden)
        self.linear2 = Linear(hidden, num_classes)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).tanh()
        x = dropout(x, p=self.p, training=self.training)

        x = self.conv2(x, edge_index).tanh()
        x = dropout(x, p=self.p, training=self.training)

        x = self.linear1(x).relu()
        x = self.linear2(x)

        return x


class GAT(torch.nn.Module):
    def __init__(self, num_classes, hidden=64, p=0.3, heads=2):
        super().__init__()
        self.p = p

        self.conv1 = GATConv((-1, -1), hidden, heads=heads, concat=False, edge_dim=1, add_self_loops=False)
        self.conv2 = GATConv((-1, -1), hidden * 2, heads=heads, concat=False, edge_dim=1, add_self_loops=False)
        self.skip1 = Linear(-1, hidden)
        self.skip2 = Linear(-1, hidden * 2)

        self.linear1 = Linear(hidden * 2, hidden)
        self.linear2 = Linear(hidden, num_classes)

    def forward(self, x, edge_index, edge_attr):
        x = (self.conv1(x, edge_index, edge_attr) + self.skip1(x)).tanh()
        x = dropout(x, p=self.p, training=self.training)

        x = (self.conv2(x, edge_index, edge_attr) + self.skip2(x)).tanh()
        x = dropout(x, p=self.p, training=self.training)

        x = self.linear1(x).relu()
        x = self.linear2(x)

        return x

## 5. Training and evaluation

We use `NeighborLoader` for mini-batch training. For each seed (`product`) node it samples `num_neighbors[k]` neighbors **for each relation** at hop `k`. This is the "neighbor selection" hyperparameter.

Within a batch, the first `batch['product'].batch_size` product nodes are the seed nodes, so the loss and the predictions use only those nodes. Because of the class imbalance, the loss can optionally use **class weights** (inverse class frequency on the training set).

We report accuracy and the **macro**-averaged precision, recall and F1, where every class counts equally. Weighted F1 is reported too.

In [14]:
def forward_model(model, batch, use_edge_attr):
    if use_edge_attr:
        return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict)
    return model(batch.x_dict, batch.edge_index_dict)


def train(model, train_loader, optimizer, criterion, epochs=5, use_edge_attr=False):
    for epoch in range(epochs):
        model.train()
        total_loss, total_examples = 0.0, 0
        start = time.time()

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            bs = batch['product'].batch_size
            out = forward_model(model, batch, use_edge_attr)['product'][:bs]
            loss = criterion(out, batch['product'].y[:bs])

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * bs
            total_examples += bs

        print(f'Epoch: {epoch:03d}, Loss: {total_loss / total_examples:.4f}, Time: {time.time() - start:.1f}s')


@torch.no_grad()
def predict(model, loader, use_edge_attr=False):
    model.eval()
    y_true, y_pred = [], []
    for batch in loader:
        batch = batch.to(device)
        bs = batch['product'].batch_size
        out = forward_model(model, batch, use_edge_attr)['product'][:bs]
        y_pred.append(out.argmax(dim=-1).cpu())
        y_true.append(batch['product'].y[:bs].cpu())
    return torch.cat(y_true).numpy(), torch.cat(y_pred).numpy()


def compute_metrics(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

In [15]:
train_counts = torch.bincount(data['product'].y[data['product'].train_mask], minlength=len(GROUPS)).float()
class_weights = (train_counts.sum() / (len(GROUPS) * train_counts))
print('class weights:', class_weights.tolist())


def run_experiment(model_type='sage', num_neighbors=(10, 10), batch_size=1024, lr=0.001,
                   epochs=3, hidden=64, weighted_loss=True, seed=42, verbose=True):
    set_seed(seed)
    use_edge_attr = model_type == 'gat'

    train_loader = NeighborLoader(data, num_neighbors=list(num_neighbors), shuffle=True,
                                  input_nodes=('product', data['product'].train_mask),
                                  batch_size=batch_size)
    test_loader = NeighborLoader(data, num_neighbors=list(num_neighbors), shuffle=False,
                                 input_nodes=('product', data['product'].test_mask),
                                 batch_size=4096)

    base_model = GraphSAGE(len(GROUPS), hidden=hidden) if model_type == 'sage' else GAT(len(GROUPS), hidden=hidden)
    model = to_hetero(base_model, data.metadata(), aggr='sum').to(device)

    # initialize the lazy (-1) parameters with one forward pass
    with torch.no_grad():
        forward_model(model, next(iter(train_loader)).to(device), use_edge_attr)

    criterion = CrossEntropyLoss(weight=class_weights.to(device) if weighted_loss else None)
    optimizer = Adam(model.parameters(), lr=lr)

    start = time.time()
    train(model, train_loader, optimizer, criterion, epochs=epochs, use_edge_attr=use_edge_attr)
    train_time = time.time() - start

    y_true, y_pred = predict(model, test_loader, use_edge_attr)
    metrics = compute_metrics(y_true, y_pred)
    metrics['train_time_s'] = train_time
    if verbose:
        print({k: round(v, 4) for k, v in metrics.items()})
    return model, metrics, (y_true, y_pred)

class weights: [0.3408142626285553, 9.125754356384277, 1.3095777034759521, 5.190344333648682]


### 5.1 Baseline model

Heterogeneous GraphSAGE with `num_neighbors=[10, 10]`, batch size 1024, Adam with `lr=0.001`, weighted loss, 3 epochs.

In [16]:
model, base_metrics, (y_true, y_pred) = run_experiment(model_type='sage', num_neighbors=(10, 10),
                                                       batch_size=1024, lr=0.001, epochs=3)

/home/kuzemano/.local/lib/python3.13/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
/home/kuzemano/.local/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/home/kuzemano/.local/lib/python3.13/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:
print('Accuracy :', accuracy_score(y_true, y_pred))
print('Precision:', precision_score(y_true, y_pred, average='macro'))
print('Recall   :', recall_score(y_true, y_pred, average='macro'))
print('F1       :', f1_score(y_true, y_pred, average='macro'))
print()
print(classification_report(y_true, y_pred, target_names=GROUPS, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')
plt.imshow(cm, cmap='Blues')
plt.xticks(range(len(GROUPS)), GROUPS)
plt.yticks(range(len(GROUPS)), GROUPS)
for i in range(len(GROUPS)):
    for j in range(len(GROUPS)):
        plt.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Normalized confusion matrix (GraphSAGE baseline)')
plt.colorbar()
plt.show()

## 6. Hyperparameter experiments

We vary one hyperparameter at a time, starting from the baseline:

* **Neighbor selection** (`num_neighbors`): `[3, 3]` (as in the exercises), `[10, 10]`, `[25, 15]`, and a single hop `[10]`
* **Training:** learning rate (`0.01` vs `0.001`), number of epochs (3 vs 8), and class-weighted vs plain cross-entropy
* **Model:** GraphSAGE vs GAT, where GAT uses the rating edge attribute

In [ ]:
configs = [
    {'name': 'baseline SAGE [10,10]', 'model_type': 'sage', 'num_neighbors': (10, 10)},
    {'name': 'SAGE [3,3]', 'model_type': 'sage', 'num_neighbors': (3, 3)},
    {'name': 'SAGE [25,15]', 'model_type': 'sage', 'num_neighbors': (25, 15)},
    {'name': 'SAGE 1-hop [10]', 'model_type': 'sage', 'num_neighbors': (10,)},
    {'name': 'SAGE lr=0.01', 'model_type': 'sage', 'num_neighbors': (10, 10), 'lr': 0.01},
    {'name': 'SAGE 8 epochs', 'model_type': 'sage', 'num_neighbors': (10, 10), 'epochs': 8},
    {'name': 'SAGE unweighted loss', 'model_type': 'sage', 'num_neighbors': (10, 10), 'weighted_loss': False},
    {'name': 'GAT (rating edge_attr) [10,10]', 'model_type': 'gat', 'num_neighbors': (10, 10)},
]

results = []
for cfg in configs:
    cfg = dict(cfg)
    name = cfg.pop('name')
    print(f'===== {name} =====')
    if name.startswith('baseline'):
        metrics = base_metrics  # already trained above with the same settings
        print({k: round(v, 4) for k, v in metrics.items()})
    else:
        _, metrics, _ = run_experiment(**cfg)
    results.append({'config': name, **metrics})

results_df = pd.DataFrame(results).set_index('config')
results_df.round(4)

In [ ]:
results_df[['accuracy', 'precision', 'recall', 'f1']].plot.barh(figsize=(9, 6))
plt.xlim(0, 1)
plt.title('Test metrics per configuration (macro-averaged precision/recall/F1)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Discussion

*Fill in the numbers from the results table above after running the notebook.*

* **Neighbor selection:** sampling more neighbors (`[25, 15]` instead of `[3, 3]`) gives each product a richer view of its similar products and reviewers. This usually raises macro-F1, but each epoch takes longer. A single hop `[10]` cannot reach the products co-rated by the same customers (`product -> customer -> product`), so it typically does worse than two hops.
* **Training:** class weighting trades some accuracy (dominated by `Book`) for higher recall on the minority classes `DVD` and `Video`, which are also the classes most often confused with each other. A larger learning rate or more epochs speeds up convergence. GAT, which uses the rating edge attribute, can be compared directly with GraphSAGE in the table.